In [1]:
import minari
dataset = minari.load_dataset('atari/tennis/expert-v0')


In [2]:
dataset.action_space

Discrete(18)

In [3]:
samples = dataset.sample_episodes(n_episodes=1)

In [4]:
import numpy as np
# split_samples = {
#     "score": [],
#     "rewards": [],
#     "actions": [],
#     "observations": []
# }
split_samples = []
for sample in samples:
    nozero_idxs = np.where(sample.rewards != 0)[0]
    start = 0
    for nonzero_idx in nozero_idxs:
        # split_samples["score"].append(sample.rewards[nonzero_idx])
        # split_samples["rewards"].append(sample.rewards[start:nonzero_idx+1])
        # split_samples["actions"].append(sample.actions[start:nonzero_idx+1])
        # split_samples["observations"].append(sample.observations[start:nonzero_idx+1])
        split_sample = {
            "score": sample.rewards[nonzero_idx],
            "rewards": sample.rewards[start:nonzero_idx+1],
            "actions": sample.actions[start:nonzero_idx+1],
            "observations": sample.observations[start:nonzero_idx+1],
            "dones": [0]*(nonzero_idx - start) + [1]
        }
        split_samples.append(split_sample)
        start = nonzero_idx + 1

In [5]:
# len(split_samples["observations"])
len(split_samples)

46

In [6]:
from dataclasses import dataclass
import random
import torch

@dataclass
class DecisionTransformerVisionDataCollator:
    return_tensors: str = "pt"
    max_len: int = 20 #subsets of the episode we use for training
    # state_dim: int = 17  # size of state space
    act_dim: int = 18  # size of action space
    max_ep_len: int = 1000 # max episode length in the dataset
    scale: float = 1000.0  # normalization of rewards/returns
    state_mean: np.array = None  # to store state means
    state_std: np.array = None  # to store state stds
    p_sample: np.array = None  # a distribution to take account trajectory lengths
    n_traj: int = 0 # to store the number of trajectories in the dataset

    def __init__(self, dataset, act_dim, image_preprocess) -> None:
        self.act_dim = act_dim
        # self.state_dim = len(dataset[0]["observations"][0])
        self.dataset = dataset
        # calculate dataset stats for normalization of states
        states = []
        traj_lens = []
        for obs in dataset:
            states.extend(obs["observations"])
            traj_lens.append(len(obs["observations"]))
        self.n_traj = len(traj_lens)
        states = np.vstack(states)
        # TODO: exchange with image normalization if needed
        self.image_preprocess = image_preprocess
        # self.state_mean, self.state_std = np.mean(states, axis=0), np.std(states, axis=0) + 1e-6
        
        traj_lens = np.array(traj_lens)
        self.p_sample = traj_lens / sum(traj_lens)

    def _discount_cumsum(self, x, gamma):
        discount_cumsum = np.zeros_like(x)
        discount_cumsum[-1] = x[-1]
        for t in reversed(range(x.shape[0] - 1)):
            discount_cumsum[t] = x[t] + gamma * discount_cumsum[t + 1]
        return discount_cumsum

    def __call__(self, features):
        batch_size = len(features)
        # this is a bit of a hack to be able to sample of a non-uniform distribution
        batch_inds = np.random.choice(
            np.arange(self.n_traj),
            size=batch_size,
            replace=True,
            p=self.p_sample,  # reweights so we sample according to timesteps
        )
        # a batch of dataset features
        s, a, r, d, rtg, timesteps, mask = [], [], [], [], [], [], []
        
        for ind in batch_inds:
            # for feature in features:
            feature = self.dataset[int(ind)]
            si = random.randint(0, len(feature["rewards"]) - 1)

            # get sequences from dataset
            # TODO: replace with image observations
            obs_img = [self.image_preprocess(obs) for obs in feature["observations"][si : si + self.max_len]]
            s.append(np.array(obs_img).reshape(1, -1, 3, obs_img[0].shape[1], obs_img[0].shape[2]))  # reshape for image observations
            one_hot_actions = np.eye(self.act_dim)[feature["actions"][si : si + self.max_len]].reshape(1, -1, self.act_dim)
            a.append(one_hot_actions)
            r.append(np.array(feature["rewards"][si : si + self.max_len]).reshape(1, -1, 1))

            d.append(np.array(feature["dones"][si : si + self.max_len]).reshape(1, -1))
            timesteps.append(np.arange(si, si + s[-1].shape[1]).reshape(1, -1))
            timesteps[-1][timesteps[-1] >= self.max_ep_len] = self.max_ep_len - 1  # padding cutoff
            rtg.append(
                self._discount_cumsum(np.array(feature["rewards"][si:]), gamma=1.0)[
                    : s[-1].shape[1]   # TODO check the +1 removed here
                ].reshape(1, -1, 1)
            )
            if rtg[-1].shape[1] < s[-1].shape[1]:
                print("if true")
                rtg[-1] = np.concatenate([rtg[-1], np.zeros((1, 1, 1))], axis=1)

            # padding and state + reward normalization
            tlen = s[-1].shape[1]
            s[-1] = np.concatenate([np.zeros((1, self.max_len - tlen, 3, s[-1].shape[3], s[-1].shape[4])), s[-1]], axis=1)
            # s[-1] = (s[-1] - self.state_mean) / self.state_std
            a[-1] = np.concatenate(
                [np.ones((1, self.max_len - tlen, self.act_dim)) * -10.0, a[-1]],
                axis=1,
            )
            r[-1] = np.concatenate([np.zeros((1, self.max_len - tlen, 1)), r[-1]], axis=1)
            d[-1] = np.concatenate([np.ones((1, self.max_len - tlen)) * 2, d[-1]], axis=1)
            rtg[-1] = np.concatenate([np.zeros((1, self.max_len - tlen, 1)), rtg[-1]], axis=1) / self.scale
            timesteps[-1] = np.concatenate([np.zeros((1, self.max_len - tlen)), timesteps[-1]], axis=1)
            mask.append(np.concatenate([np.zeros((1, self.max_len - tlen)), np.ones((1, tlen))], axis=1))

        s = torch.from_numpy(np.concatenate(s, axis=0)).float()
        a = torch.from_numpy(np.concatenate(a, axis=0)).float()
        r = torch.from_numpy(np.concatenate(r, axis=0)).float()
        d = torch.from_numpy(np.concatenate(d, axis=0))
        rtg = torch.from_numpy(np.concatenate(rtg, axis=0)).float()
        timesteps = torch.from_numpy(np.concatenate(timesteps, axis=0)).long()
        mask = torch.from_numpy(np.concatenate(mask, axis=0)).float()

        return {
            "states": s,
            "actions": a,
            "rewards": r,
            "returns_to_go": rtg,
            "timesteps": timesteps,
            "attention_mask": mask,
        }

In [7]:
from transformers import DecisionTransformerConfig, DecisionTransformerModel
from torchvision import models
from torch import nn
class VisionDT(DecisionTransformerModel):
    def __init__(self, config, resnet_weights=None):
        super().__init__(config)
        # replace the state embedding with a vision model
        pretrained_resnet50 = models.resnet50(weights=resnet_weights)
        resnet_feature_size = pretrained_resnet50.fc.in_features
        pretrained_resnet50.fc = nn.Linear(resnet_feature_size, config.hidden_size)
        self.embed_state = nn.Sequential(
            nn.Flatten(start_dim=0, end_dim=1),
            pretrained_resnet50,
            nn.Unflatten(dim=0, unflattened_size=(-1, 20)),
        )

    def forward(self, **kwargs):
        output = super().forward(**kwargs)
        # add the DT loss
        action_preds = output[1]
        action_targets = kwargs["actions"]
        attention_mask = kwargs["attention_mask"]
        act_dim = action_preds.shape[2]
        action_preds = action_preds.reshape(-1, act_dim)[attention_mask.reshape(-1) > 0]
        action_targets = action_targets.reshape(-1, act_dim)[attention_mask.reshape(-1) > 0]
        loss = torch.nn.CrossEntropyLoss()(action_preds, torch.argmax(action_targets, dim=1))

        return {"loss": loss}

    def original_forward(self, **kwargs):
        return super().forward(**kwargs)

In [8]:
resnet_weights = models.ResNet50_Weights.IMAGENET1K_V1
config = DecisionTransformerConfig(act_dim=18)
model = VisionDT(config, resnet_weights=resnet_weights)

In [9]:
from torchvision import transforms
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    resnet_weights.transforms(),
])
collator = DecisionTransformerVisionDataCollator(split_samples, act_dim=18, image_preprocess=preprocess)

In [10]:
preprocess

Compose(
    ToPILImage()
    ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)
)

In [11]:
from transformers import Trainer, TrainingArguments
training_args = TrainingArguments(
    output_dir="output/tennis/",
    remove_unused_columns=False,
    num_train_epochs=120,
    per_device_train_batch_size=8,
    learning_rate=1e-4,
    weight_decay=1e-4,
    warmup_ratio=0.1,
    optim="adamw_torch",
    max_grad_norm=0.25,
    logging_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset= split_samples,
    data_collator=collator,
)

trainer.train()

/opt/miniconda3/envs/sdsc6007/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
6,2.265400


KeyboardInterrupt: 